# Phase 1 — Day 2: NumPy Vectorization

**Date:** 2026-04-22

NumPy is the backbone of numerical computing in Python. Today you'll learn how to think in arrays instead of loops.

## Learning Objectives
- Create and manipulate NumPy arrays
- Understand dtypes and why they matter
- Replace Python loops with vectorized operations
- Use broadcasting to operate on arrays of different shapes
- Work with the `axis` parameter in aggregation functions

In [1]:
# Setup
import numpy as np
import time

print(f"NumPy version: {np.__version__}")
print("Ready to go!")

NumPy version: 2.4.4
Ready to go!


In [2]:
# Sample data: simulated daily sales for 5 stores over 7 days
np.random.seed(42)

daily_sales = np.random.randint(100, 500, size=(5, 7))
store_names = ["Istanbul", "Ankara", "Izmir", "Bursa", "Antalya"]
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

print("Daily sales matrix (5 stores x 7 days):")
print(daily_sales)
print(f"\nShape: {daily_sales.shape}")
print(f"Stores: {store_names}")
print(f"Days: {day_names}")

Daily sales matrix (5 stores x 7 days):
[[202 448 370 206 171 288 120]
 [202 221 314 430 187 472 199]
 [459 251 230 249 408 357 443]
 [393 485 291 376 260 413 121]
 [352 335 444 148 158 269 287]]

Shape: (5, 7)
Stores: ['Istanbul', 'Ankara', 'Izmir', 'Bursa', 'Antalya']
Days: ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']


---
## 1. Arrays and How to Create Them

A NumPy array is a grid of values, all the same type. Unlike Python lists, arrays are stored in contiguous memory, which makes math on them very fast.

You can create arrays from lists, from scratch using helper functions, or by generating random data. Let's see the main ways.

In [9]:
# Creating arrays

# From a Python list
a = np.array([1, 2, 3, 4, 5])
print("From list:", a)

# 2D array from nested lists
b = np.array([[1, 2, 3], [4, 5, 6]])
print("2D array:\n", b)

# Common creation functions
print("zeros:\n", np.zeros(5))
print("ones:\n", np.ones((2, 3)))
print("arange:\n", np.arange(0, 10, 2))       # start, stop, step
print("linspace:\n", np.linspace(0, 1, 5))     # start, stop, num_points
print("eye:\n", np.eye(3))                   # identity matrix
print("full:\n", np.full((2, 3), 7))           # fill with a value

From list: [1 2 3 4 5]
2D array:
 [[1 2 3]
 [4 5 6]]
zeros:
 [0. 0. 0. 0. 0.]
ones:
 [[1. 1. 1.]
 [1. 1. 1.]]
arange:
 [0 2 4 6 8]
linspace:
 [0.   0.25 0.5  0.75 1.  ]
eye:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
full:
 [[7 7 7]
 [7 7 7]]


In [ ]:
# Array attributes you'll use all the time
arr = np.random.randint(1, 100, size=(3, 4))
print("Array:\n", arr)
print(f"\nshape: {arr.shape}")      # dimensions
print(f"ndim: {arr.ndim}")          # number of axes
print(f"size: {arr.size}")          # total element count
print(f"dtype: {arr.dtype}")        # data type
print(f"itemsize: {arr.itemsize} bytes")  # bytes per element

---
## 2. Dtypes and Why They Matter

Every element in a NumPy array has the same data type (dtype). This is what makes arrays fast. NumPy doesn't need to check "is this an int or a string?" for each element.

Common dtypes: `int32`, `int64`, `float32`, `float64`, `bool`, `object` (avoid this one). Choosing the right dtype saves memory and speeds up computation. For example, `float32` uses half the memory of `float64`.

In [13]:
# Dtypes in action

# NumPy picks a dtype automatically
a = np.array([1, 2, 3])
print(f"Integers: dtype = {a.dtype}")

b = np.array([1.0, 2.0, 3.0])
print(f"Floats: dtype = {b.dtype}")

# You can specify the dtype
c = np.array([1, 2, 3], dtype=np.float32)
print(f"Forced float32: dtype = {c.dtype}")

# Memory comparison
big_f64 = np.zeros(1_000_000, dtype=np.float64)
big_f32 = np.zeros(1_000_000, dtype=np.float32)
big_int32 = np.zeros(1_000_000, dtype=np.int32)
print(f"\nfloat64 array: {big_f64.nbytes / 1e6:.1f} MB")
print(f"float32 array: {big_f32.nbytes / 1e6:.1f} MB")
print(f"int32 array: {big_int32.nbytes / 1e6:.1f} MB")
print("Half the memory!")

Integers: dtype = int64
Floats: dtype = float64
Forced float32: dtype = float32

float64 array: 8.0 MB
float32 array: 4.0 MB
int32 array: 4.0 MB
Half the memory!


In [14]:
# Casting between dtypes
a = np.array([1.7, 2.3, 3.9])
print(f"Original (float64): {a}")

# astype creates a new array with the new dtype
b = a.astype(np.int32)
print(f"Cast to int32: {b}")  # Notice: it truncates, doesn't round!

c = a.astype(np.int32).astype(np.float64)
print(f"Back to float64: {c}")  # The decimal part is gone forever

Original (float64): [1.7 2.3 3.9]
Cast to int32: [1 2 3]
Back to float64: [1. 2. 3.]


---
## 3. Vectorization: Why Loops Are (Usually) Bad

Vectorization means applying an operation to an entire array at once instead of looping through elements. NumPy operations run in optimized C code under the hood, so vectorized code is typically 10-100x faster than Python loops.

The rule of thumb: if you're writing a `for` loop over array elements, there's probably a NumPy function that does it faster.

In [22]:
# Speed comparison: loop vs vectorized

size = 1_000_000_0
a = np.random.rand(size)
b = np.random.rand(size)

# Python loop
start = time.time()
result_loop = [a[i] + b[i] for i in range(size)]
loop_time = time.time() - start

# Vectorized
start = time.time()
result_vec = a + b
vec_time = time.time() - start

print(f"Python loop: {loop_time:.4f} seconds")
print(f"Vectorized:  {vec_time:.4f} seconds")
print(f"Speedup:     {loop_time / vec_time:.0f}x faster!")

Python loop: 1.3662 seconds
Vectorized:  0.0036 seconds
Speedup:     382x faster!


In [26]:
# Common vectorized operations

prices = np.array([10.5, 20.0, 35.5, 15.0, 42.0])

# Element-wise math
print("Double prices:", prices * 2)
print("Add tax (18%):", prices * 1.18)
print("Square root:  ", np.sqrt(prices))

# Comparisons return boolean arrays
print("\nPrices > 20:", prices > 20)
print("Count > 20: ", np.sum(prices > 20))

# Boolean indexing (filtering)
expensive = prices[prices > 20]
print("Expensive items:", expensive)

# Replace values conditionally
capped = np.where(prices > 15, 30, prices)  # cap at 30
print("Capped at 30:  ", capped)

Double prices: [21. 40. 71. 30. 84.]
Add tax (18%): [12.39 23.6  41.89 17.7  49.56]
Square root:   [3.24037035 4.47213595 5.95818764 3.87298335 6.4807407 ]

Prices > 20: [False False  True False  True]
Count > 20:  2
Expensive items: [35.5 42. ]
Capped at 30:   [10.5 30.  30.  15.  30. ]


---
## 4. Broadcasting

Broadcasting is NumPy's way of handling operations between arrays of different shapes. Instead of requiring you to manually reshape or tile arrays, NumPy stretches the smaller array to match the larger one.

The broadcasting rules are simple:
1. If the arrays have different numbers of dimensions, pad the smaller shape with 1s on the left.
2. Arrays with size 1 along a dimension act as if they had the size of the largest array in that dimension.
3. If sizes disagree and neither is 1, you get an error.

In [ ]:
# Broadcasting example 1: scalar + array
# The scalar "broadcasts" to match the array shape
a = np.array([1, 2, 3, 4])
print("a + 10 =", a + 10)  # 10 is broadcast to [10, 10, 10, 10]

# Broadcasting example 2: row + matrix
# A (1,3) array broadcasts across all rows of a (3,3) matrix
matrix = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])
row = np.array([10, 20, 30])

print("\nMatrix:\n", matrix)
print("Row:", row)
print("Matrix + Row:\n", matrix + row)

In [39]:
# Broadcasting example 3: column + row = 2D grid
# This is a powerful trick for creating grids

col = np.array([[1], [2], [3]])  # shape (3, 1)
row = np.array([10, 20, 30])     # shape (3,)

print(f"col shape: {col.shape}")
print(f"row shape: {row.shape}")
print(f"\ncol:\n{col}")
print(f"\nrow:\n{row}")
print(f"\ncol + row (creates a 3x3 grid):\n{col + row}")

# Real-world use: normalize each column of our sales data
print("\n--- Normalizing daily_sales ---")
print("Original daily_sales:\n", daily_sales,"\n")
col_means = daily_sales.mean(axis=0)  # mean per day (across stores)
col_stds = daily_sales.std(axis=0)    # std per day
normalized = (daily_sales - col_means) / col_stds
print("Normalized daily_sales:\n", normalized)
print("Normalized first row:", normalized[0].round(2))
print("Normalized first row:", normalized[1].round(2))

col shape: (3, 1)
row shape: (3,)

col:
[[1]
 [2]
 [3]]

row:
[10 20 30]

col + row (creates a 3x3 grid):
[[11 21 31]
 [12 22 32]
 [13 23 33]]

--- Normalizing daily_sales ---
Original daily_sales:
 [[202 448 370 206 171 288 120]
 [202 221 314 430 187 472 199]
 [459 251 230 249 408 357 443]
 [393 485 291 376 260 413 121]
 [352 335 444 148 158 269 287]] 

Normalized daily_sales:
 [[-1.15611744  0.95803725  0.55351951 -0.719068   -0.71064117 -0.9456082
  -0.9403841 ]
 [-1.15611744 -1.21670731 -0.21755244  1.40588229 -0.53784088  1.47767744
  -0.28871442]
 [ 1.32818174 -0.92929613 -1.37416038 -0.31115344  1.84896304 -0.03687609
   1.72403753]
 [ 0.69019051  1.31251104 -0.53424271  0.89361749  0.25056041  0.70064563
  -0.93213512]
 [ 0.29386263 -0.12454484  1.57243603 -1.26927834 -0.8510414  -1.19583878
   0.43719612]]
Normalized first row: [-1.16  0.96  0.55 -0.72 -0.71 -0.95 -0.94]
Normalized first row: [-1.16 -1.22 -0.22  1.41 -0.54  1.48 -0.29]


---
## 5. The `axis` Parameter

This trips up almost everyone at first. When you call `np.sum(arr, axis=0)`, you're summing **along** axis 0 (collapsing the rows). The result has one fewer dimension.

Think of it this way:
- `axis=0` means "go down the rows" (result has one value per column)
- `axis=1` means "go across the columns" (result has one value per row)
- `axis=None` (default) means "flatten everything and compute one number"

In [40]:
# axis in action with our daily_sales (5 stores x 7 days)

print("daily_sales shape:", daily_sales.shape)
print()

# Total sales across ALL stores and days
total = daily_sales.sum()  # axis=None by default
print(f"Grand total: {total}")

# Total sales PER DAY (sum across stores, axis=0)
per_day = daily_sales.sum(axis=0)
print(f"\nPer day (axis=0): {per_day}")
print(f"Shape: {per_day.shape}")  # (7,) - one value per day

# Total sales PER STORE (sum across days, axis=1)
per_store = daily_sales.sum(axis=1)
print(f"\nPer store (axis=1): {per_store}")
print(f"Shape: {per_store.shape}")  # (5,) - one value per store

# Same works for mean, std, min, max, argmin, argmax
print(f"\nBest day per store (argmax axis=1): {daily_sales.argmax(axis=1)}")
print(f"Highest-selling store per day (argmax axis=0): {daily_sales.argmax(axis=0)}")

daily_sales shape: (5, 7)

Grand total: 10559

Per day (axis=0): [1608 1740 1649 1409 1184 1799 1170]
Shape: (7,)

Per store (axis=1): [1805 2025 2397 2339 1993]
Shape: (5,)

Best day per store (argmax axis=1): [1 5 0 1 2]
Highest-selling store per day (argmax axis=0): [2 3 4 1 2 1 2]


In [43]:
# keepdims=True: preserves the original number of dimensions
# This is useful for broadcasting after aggregation

row_means = daily_sales.mean(axis=1)
print(f"Without keepdims: shape = {row_means.shape}")  # (5,)

row_means_kd = daily_sales.mean(axis=1, keepdims=True)
print(f"With keepdims:    shape = {row_means_kd.shape}")  # (5, 1)

# Now you can subtract the mean from each row easily
centered = daily_sales - row_means_kd  # broadcasting works!
print(f"\nCentered row 0 mean: {centered[0].mean():.10f}")  # ~0

Without keepdims: shape = (5,)
With keepdims:    shape = (5, 1)

Centered row 0 mean: 0.0000000000


---
## Tricky Bits

These are the gotchas that catch people off guard. Read through these carefully.

In [44]:
# GOTCHA 1: Views vs Copies
# Slicing an array creates a VIEW, not a copy. Modifying the view changes the original!

original = np.array([1, 2, 3, 4, 5])
view = original[1:4]
view[0] = 999

print("original after modifying view:", original)  # [1, 999, 3, 4, 5] !!
print("The original changed! This is a view, not a copy.")

# To make a true copy, use .copy()
original2 = np.array([1, 2, 3, 4, 5])
safe_copy = original2[1:4].copy()
safe_copy[0] = 999
print("\noriginal2 after modifying copy:", original2)  # [1, 2, 3, 4, 5]
print("Original is safe. This is a copy.")

original after modifying view: [  1 999   3   4   5]
The original changed! This is a view, not a copy.

original2 after modifying copy: [1 2 3 4 5]
Original is safe. This is a copy.


In [51]:
# GOTCHA 2: Broadcasting shape mismatch
# When shapes don't align, you get an error

a = np.array([1, 2, 3])      # shape (3,)
b = np.array([1, 2, 3, 4])   # shape (4,)

try:
    result = a + b
except ValueError as e:
    print(f"Error: {e}")
    print("Shapes (3,) and (4,) can't be broadcast together!")

# GOTCHA 3: Integer division surprise
a = np.array([1, 2, 3], dtype=np.int64)
b = a / 2
print(f"\nOriginal int array: {a}")
print(f"int array / 2 = {b}")
print(f"dtype is now: {b.dtype}")  # float64! Division always returns float

# But floor division stays integer
c = a // 2
print(f"int array // 2 = {c}")
print(f"dtype: {c.dtype}")  # still int64

Error: operands could not be broadcast together with shapes (3,) (4,) 
Shapes (3,) and (4,) can't be broadcast together!

Original int array: [1 2 3]
int array / 2 = [0.5 1.  1.5]
dtype is now: float64
int array // 2 = [0 1 1]
dtype: int64


In [ ]:
# GOTCHA 4: Mixing ints and strings in an array
# NumPy will silently convert everything to strings (dtype='<U21')

mixed = np.array([1, "hello", 3.14])
print(f"Mixed array: {mixed}")
print(f"dtype: {mixed.dtype}")  # Unicode string!
print(f"mixed[0] + 1 would fail because '1' is now a string")

try:
    result = mixed[0] + 1
except TypeError as e:
    print(f"Error: {e}")

---
## Trick Questions

Test your understanding. Try to answer before revealing the solution.

<details>
<summary><strong>Q1: What does np.array([1, 2, 3]) * np.array([True, False, True]) return?</strong></summary>

`array([1, 0, 3])`. Booleans are treated as 0 and 1 in arithmetic. `True` = 1, `False` = 0. So this is element-wise multiplication: `[1*1, 2*0, 3*1]`.
</details>

<details>
<summary><strong>Q2: What is the shape of np.zeros((3,)) + np.zeros((1, 3))?</strong></summary>

`(1, 3)`. The first array has shape `(3,)` which broadcasts to `(1, 3)`. Adding two `(1, 3)` arrays gives `(1, 3)`.
</details>

<details>
<summary><strong>Q3: If a = np.array([1, 2, 3]) and b = a, does modifying b change a?</strong></summary>

Yes! `b = a` does NOT copy the array. Both variables point to the same data. Use `b = a.copy()` if you want independence.
</details>

<details>
<summary><strong>Q4: What does np.arange(0.1, 0.4, 0.1) return? Is the length 3 or 4?</strong></summary>

Usually `array([0.1, 0.2, 0.3])`, length 3. But due to floating-point precision, it could sometimes include 0.4! That's why `np.linspace` is safer when you need exact endpoint control.
</details>

<details>
<summary><strong>Q5: What happens if you do np.array([1, 2, 3])[5]?</strong></summary>

`IndexError: index 5 is out of bounds for axis 0 with size 3`. NumPy does bounds checking, unlike C arrays.
</details>

---
## Exercises

Fill in the `___` blanks and run each cell. The `assert` statements will tell you if you got it right.

In [58]:
# Exercise 1: Create an array of even numbers from 2 to 20 (inclusive)
evens = np.arange(2, 21, 2)
assert len(evens) == 10
assert evens[0] == 2 and evens[-1] == 20
print("Exercise 1 passed!", evens)

Exercise 1 passed! [ 2  4  6  8 10 12 14 16 18 20]


In [59]:
# Exercise 2: Create a 4x4 identity matrix using a NumPy function
identity = np.eye(4,4)
assert identity.shape == (4, 4)
assert identity[0, 0] == 1.0 and identity[0, 1] == 0.0
print("Exercise 2 passed!\n", identity)

Exercise 2 passed!
 [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


In [60]:
# Exercise 3: Convert Celsius to Fahrenheit (vectorized, no loops!)
# Formula: F = C * 9/5 + 32
celsius = np.array([0, 20, 37, 100])
fahrenheit = celsius*9/5 + 32
assert np.allclose(fahrenheit, [32, 68, 98.6, 212])
print("Exercise 3 passed!", fahrenheit)

Exercise 3 passed! [ 32.   68.   98.6 212. ]


In [61]:
# Exercise 4: Count how many values in this array are greater than 50
data = np.array([12, 55, 38, 72, 91, 43, 67, 29, 85, 50])
count = np.sum(data > 50)
assert count == 5
print(f"Exercise 4 passed! {count} values are > 50")

Exercise 4 passed! 5 values are > 50


In [63]:
# Exercise 5: Find the mean of each ROW in this matrix
matrix = np.array([[10, 20, 30],
                   [40, 50, 60],
                   [70, 80, 90]])
row_means = matrix.mean(axis=1)
assert np.allclose(row_means, [20, 50, 80])
print("Exercise 5 passed!", row_means)

Exercise 5 passed! [20. 50. 80.]


In [64]:
# Exercise 6: Replace all negative values with 0 using np.where
scores = np.array([85, -10, 92, -5, 78, -20, 95])
cleaned = np.where(scores < 0, 0, scores)
assert np.all(cleaned >= 0)
assert cleaned[1] == 0 and cleaned[0] == 85
print("Exercise 6 passed!", cleaned)

Exercise 6 passed! [85  0 92  0 78  0 95]


In [65]:
# Exercise 7: min max scaler this array to range [0, 1]
# Formula: (x - min) / (max - min)
raw = np.array([10, 30, 50, 70, 90])
normalized = (raw - raw.min()) / (raw.max() - raw.min())
assert normalized[0] == 0.0 and normalized[-1] == 1.0
print("Exercise 7 passed!", normalized)

Exercise 7 passed! [0.   0.25 0.5  0.75 1.  ]


### Exercise Solutions

<details>
<summary>Click to reveal all solutions</summary>

**Exercise 1:** `np.arange(2, 21, 2)` (or `np.arange(2, 22, 2)`)

**Exercise 2:** `np.eye(4)`

**Exercise 3:** `celsius * 9/5 + 32`

**Exercise 4:** `np.sum(data > 50)`

**Exercise 5:** `matrix.mean(axis=1)`

**Exercise 6:** `np.where(scores >= 0, scores, 0)` or `np.where(scores < 0, 0, scores)`

**Exercise 7:** `(raw - raw.min()) / (raw.max() - raw.min())`
</details>

---
## Cumulative Review Exercises

These exercises cover **Day 1 (Pandas Essentials)** to keep those skills fresh.

In [69]:
import pandas as pd

# Sample data for review exercises
review_df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "department": ["Sales", "Engineering", "Sales", "Engineering", "Sales"],
    "salary": [55000, 72000, 48000, 85000, 62000],
    "years": [3, 7, 1, 10, 5]
})
print(review_df)

      name   department  salary  years
0    Alice        Sales   55000      3
1      Bob  Engineering   72000      7
2  Charlie        Sales   48000      1
3    Diana  Engineering   85000     10
4      Eve        Sales   62000      5


In [70]:
# Review 1: Filter for employees with salary > 60000 using boolean indexing
high_earners = review_df[review_df["salary"] > 60000]
assert len(high_earners) == 3
print("Review 1 passed!\n", high_earners)

Review 1 passed!
     name   department  salary  years
1    Bob  Engineering   72000      7
3  Diana  Engineering   85000     10
4    Eve        Sales   62000      5


In [72]:
# Review 2: Use .loc to get the name and salary of the row at index 2
result = review_df.loc[2, ["name", "salary"]]
assert result["name"] == "Charlie"
print("Review 2 passed!", result.values)

Review 2 passed! ['Charlie' np.int64(48000)]


In [73]:
# Review 3: Use groupby to find the mean salary per department
dept_avg = review_df.groupby("department")["salary"].mean()
assert dept_avg["Engineering"] == 78500.0
print("Review 3 passed!\n", dept_avg)

Review 3 passed!
 department
Engineering    78500.0
Sales          55000.0
Name: salary, dtype: float64


In [77]:
# Review 4: Use .iloc to get the first 3 rows and the first 2 columns
subset = review_df.iloc[0:3, 0:2]
assert subset.shape == (3, 2)
print("Review 4 passed!\n", subset)

Review 4 passed!
       name   department
0    Alice        Sales
1      Bob  Engineering
2  Charlie        Sales


In [78]:
# Review 5: Create a new column "salary_k" that is salary divided by 1000
review_df["salary_k"] = review_df["salary"] / 1000
assert review_df["salary_k"].iloc[0] == 55.0
print("Review 5 passed!\n", review_df)

Review 5 passed!
       name   department  salary  years  salary_k
0    Alice        Sales   55000      3      55.0
1      Bob  Engineering   72000      7      72.0
2  Charlie        Sales   48000      1      48.0
3    Diana  Engineering   85000     10      85.0
4      Eve        Sales   62000      5      62.0


### Cumulative Review Solutions

<details>
<summary>Click to reveal all solutions</summary>

**Review 1:** `review_df[review_df["salary"] > 60000]`

**Review 2:** `review_df.loc[2, ["name", "salary"]]`

**Review 3:** `review_df.groupby("department")["salary"].mean()`

**Review 4:** `review_df.iloc[0:3, 0:2]` or `review_df.iloc[:3, :2]`

**Review 5:** `review_df["salary_k"] = review_df["salary"] / 1000`
</details>

---
## Cheat Sheet

In [80]:
cheat_sheet = """
╔══════════════════════════════════════════════════════════════╗
║                  NUMPY CHEAT SHEET - DAY 2                   ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  CREATE ARRAYS                                               ║
║    np.array([1,2,3])         from list                       ║
║    np.zeros((r,c))           all zeros                       ║
║    np.ones((r,c))            all ones                        ║
║    np.arange(start,stop,step) like range()                   ║
║    np.linspace(start,stop,n) n evenly spaced                 ║
║    np.eye(n)                 identity matrix                 ║
║    np.random.rand(r,c)       uniform [0,1)                   ║
║    np.random.randint(lo,hi,size) random integers             ║
║                                                              ║
║  ATTRIBUTES                                                  ║
║    .shape  .ndim  .size  .dtype  .nbytes                     ║
║                                                              ║
║  DTYPES                                                      ║
║    int32, int64, float32, float64, bool                      ║
║    .astype(np.float32)       convert dtype                   ║
║                                                              ║
║  VECTORIZED OPS                                              ║
║    a + b, a * b, a ** 2      element-wise math               ║
║    np.sqrt(a), np.log(a)     ufuncs                          ║
║    a > 5                     boolean array                   ║
║    a[a > 5]                  boolean indexing                ║
║    np.where(cond, x, y)      conditional replace             ║
║                                                              ║
║  AGGREGATIONS + AXIS                                         ║
║    .sum() .mean() .std() .min() .max()                       ║
║    axis=0  -> collapse rows (per column)                     ║
║    axis=1  -> collapse cols (per row)                        ║
║    axis=None -> everything (default)                         ║
║    keepdims=True -> keep original ndim                       ║
║                                                              ║
║  BROADCASTING RULES                                          ║
║    1. Pad smaller shape with 1s on the left                  ║
║    2. Size-1 dims stretch to match                           ║
║    3. Non-1 mismatched dims = error                          ║
║                                                              ║
║  WATCH OUT                                                   ║
║    Slices are VIEWS (use .copy() for safety)                 ║
║    Mixed types -> silent conversion to broadest type         ║
║    / always returns float; // for integer division           ║
╚══════════════════════════════════════════════════════════════╝
"""
print(cheat_sheet)


╔══════════════════════════════════════════════════════════════╗
║                  NUMPY CHEAT SHEET - DAY 2                   ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  CREATE ARRAYS                                               ║
║    np.array([1,2,3])         from list                       ║
║    np.zeros((r,c))           all zeros                       ║
║    np.ones((r,c))            all ones                        ║
║    np.arange(start,stop,step) like range()                   ║
║    np.linspace(start,stop,n) n evenly spaced                 ║
║    np.eye(n)                 identity matrix                 ║
║    np.random.rand(r,c)       uniform [0,1)                   ║
║    np.random.randint(lo,hi,size) random integers             ║
║                                                              ║
║  ATTRIBUTES                                                  ║
║    .shape  .ndim  .siz

---
## Next up: Day 3 — DataCleaningAndIO

You'll learn how to handle missing values, remove duplicates, convert dtypes, and read/write files in CSV, JSON, and Excel formats. The messy real-world data starts here!